# V18 BCR/TCR Data Deep Scan + h5ad Analysis
**Date:** 2026-03-14
**Purpose:** 
1. Scan Zhang Gut supplementary folder and extracted files folder
2. Analyze h5ad BCR/TCR column structure in detail
3. Produce tissue-separated × stage × donor summary

In [ ]:
# Cell 1: Mount Drive & scan both folders
from google.colab import drive
drive.mount('/content/drive')
import os, glob
import pandas as pd
import numpy as np

# --- Folder 1: Gut supplementary data ---
supp_root = '/content/drive/MyDrive/ITLAS/data/raw/gut_2021'
# Also try parent
supp_parent = '/content/drive/MyDrive/ITLAS/data/raw'

# --- Folder 2: Extracted files ---
ext_root = '/content/drive/MyDrive/ITLAS/data/raw/zips'

def scan_folder(path, label, max_depth=3):
    print(f'\n{"="*60}')
    print(f'SCANNING: {label}')
    print(f'Path: {path}')
    print(f'{"="*60}')
    if not os.path.exists(path):
        print(f'  ⚠️ Path does not exist!')
        # Try alternatives
        alternatives = [
            path,
            path.replace('raw/gut_2021', 'raw'),
            path.replace('raw/zips', 'raw'),
        ]
        for alt in alternatives:
            if os.path.exists(alt) and alt != path:
                print(f'  → Found alternative: {alt}')
                path = alt
                break
        else:
            return []
    
    all_files = []
    for root, dirs, files in os.walk(path):
        depth = root.replace(path, '').count(os.sep)
        if depth > max_depth:
            continue
        indent = '  ' * (depth + 1)
        rel = os.path.relpath(root, path)
        if rel != '.':
            print(f'{indent}📁 {os.path.basename(root)}/')
        for f in sorted(files):
            fpath = os.path.join(root, f)
            fsize = os.path.getsize(fpath) / 1024
            unit = 'KB'
            if fsize > 1024:
                fsize /= 1024
                unit = 'MB'
            if fsize > 1024:
                fsize /= 1024
                unit = 'GB'
            ext = os.path.splitext(f)[1].lower()
            flag = ''
            fl = f.lower()
            if any(kw in fl for kw in ['bcr','tcr','vdj','clone','contig','repertoire']):
                flag = ' ⭐ BCR/TCR'
            elif ext in ['.csv', '.tsv', '.txt']:
                flag = ' 📊'
            print(f'{indent}  {f} ({fsize:.1f} {unit}){flag}')
            all_files.append({'path': fpath, 'name': f, 'size_kb': os.path.getsize(fpath)/1024, 'ext': ext})
    return all_files

# Scan all potential paths
supp_files = scan_folder(supp_root, 'Zhang Gut 2021 Supplementary')
if not supp_files:
    supp_files = scan_folder(supp_parent, 'Raw data folder (parent)')

ext_files = scan_folder(ext_root, 'Extracted files')
if not ext_files:
    # Try finding the actual location
    print('\nSearching entire ITLAS/data for CSV/TSV files with BCR/TCR keywords...')
    for root, dirs, files in os.walk('/content/drive/MyDrive/ITLAS/data'):
        for f in files:
            fl = f.lower()
            if any(kw in fl for kw in ['bcr','tcr','vdj','clone','contig']):
                fpath = os.path.join(root, f)
                print(f'  Found: {fpath} ({os.path.getsize(fpath)/1024:.1f} KB)')

In [ ]:
# Cell 2: Preview any CSV/TSV files found — especially BCR/TCR related
all_found = supp_files + ext_files
csv_files = [f for f in all_found if f['ext'] in ['.csv', '.tsv', '.txt']]

print(f'Total CSV/TSV/TXT files found: {len(csv_files)}')
print()

for finfo in csv_files:
    fpath = finfo['path']
    fname = finfo['name']
    print(f'\n{"─"*60}')
    print(f'📄 {fname} ({finfo["size_kb"]:.1f} KB)')
    print(f'   {fpath}')
    try:
        # Try reading first few rows
        sep = '\t' if finfo['ext'] == '.tsv' else ','
        df = pd.read_csv(fpath, sep=sep, nrows=5)
        print(f'   Columns ({len(df.columns)}): {list(df.columns)}')
        print(f'   Shape preview: {df.shape}')
        # Check if BCR/TCR related
        cols_lower = [c.lower() for c in df.columns]
        is_vdj = any(kw in ' '.join(cols_lower) for kw in 
                     ['bcr','tcr','vdj','clone','cdr3','v_gene','j_gene',
                      'chain','contig','barcode','igh','igk','igl','tra','trb'])
        if is_vdj:
            print(f'   ⭐ BCR/TCR RELATED!')
            # Read full file for row count
            df_full = pd.read_csv(fpath, sep=sep)
            print(f'   Full shape: {df_full.shape}')
            print(f'   First 3 rows:')
            print(df_full.head(3).to_string())
    except Exception as e:
        print(f'   Error reading: {e}')
        # Try with different separators
        try:
            df = pd.read_csv(fpath, sep='\t', nrows=3)
            print(f'   (tab-sep) Columns: {list(df.columns)}')
        except:
            pass

In [ ]:
# Cell 3: Deep analysis of h5ad BCR/TCR columns
import scanpy as sc

DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
adata = sc.read_h5ad(DATA_PATH, backed='r')
obs = adata.obs.copy()

# Derive donor column
obs['donor'] = obs['sample'].astype(str).str.split('_').str[1]

print(f'Total cells: {len(obs):,}')
print(f'\n{"="*60}')
print('BCR COLUMNS DEEP ANALYSIS')
print(f'{"="*60}')

# BCR overview
bcr_mask = obs['BCR_clone.id'].notna()
print(f'\nBCR+ cells: {bcr_mask.sum():,} / {len(obs):,} ({bcr_mask.mean()*100:.1f}%)')

# BCR by Stage × Tissue
print(f'\n--- BCR+ cells by Stage × Tissue ---')
bcr_xt = obs.groupby(['Stage', 'tissue'], observed=True).agg(
    total_cells=('BCR_clone.id', 'size'),
    bcr_cells=('BCR_clone.id', lambda x: x.notna().sum()),
).reset_index()
bcr_xt['pct'] = (bcr_xt['bcr_cells'] / bcr_xt['total_cells'] * 100).round(1)
print(bcr_xt.to_string(index=False))

# BCR Isotype distribution by Stage × Tissue
print(f'\n--- BCR Isotype (BCR_CType) by Stage × Tissue ---')
iso = obs[obs['BCR_CType'].notna()].groupby(
    ['Stage', 'tissue', 'BCR_CType'], observed=True
).size().reset_index(name='count')
iso_pivot = iso.pivot_table(
    index=['Stage', 'tissue'], columns='BCR_CType', values='count', fill_value=0
)
iso_pivot['total'] = iso_pivot.sum(axis=1)
print(iso_pivot.to_string())

# Isotype percentages
print(f'\n--- BCR Isotype % by Stage × Tissue ---')
iso_pct = iso_pivot.div(iso_pivot['total'], axis=0).drop(columns='total') * 100
print(iso_pct.round(1).to_string())

In [ ]:
# Cell 4: Donor-level BCR metrics (IT-oriented)
print(f'{"="*60}')
print('DONOR-LEVEL BCR METRICS — IT-Oriented')
print(f'{"="*60}')

def calc_donor_bcr_metrics(obs_df, tissue_val):
    """Calculate per-donor BCR metrics for a given tissue."""
    sub = obs_df[obs_df['tissue'] == tissue_val].copy()
    results = []
    for (stage, donor), grp in sub.groupby(['Stage', 'donor'], observed=True):
        n_total = len(grp)
        bcr_cells = grp['BCR_clone.id'].notna().sum()
        if bcr_cells == 0:
            results.append({
                'Stage': stage, 'donor': donor, 'tissue': tissue_val,
                'n_total': n_total, 'n_bcr': 0, 'pct_bcr': 0,
                'n_unique_clones': 0, 'clonality': 0,
                'pct_IgM': np.nan, 'pct_IgG': np.nan, 'pct_IgA': np.nan,
                'pct_singleton': np.nan,
            })
            continue
        
        bcr_grp = grp[grp['BCR_clone.id'].notna()]
        clone_counts = bcr_grp['BCR_clone.id'].value_counts()
        n_unique = len(clone_counts)
        n_singleton = (clone_counts == 1).sum()
        
        # Clonality (1 - normalized Shannon entropy)
        if n_unique > 1:
            freqs = clone_counts.values / clone_counts.sum()
            entropy = -np.sum(freqs * np.log2(freqs))
            max_entropy = np.log2(n_unique)
            clonality = 1 - (entropy / max_entropy) if max_entropy > 0 else 0
        else:
            clonality = 0
        
        # Isotype distribution
        iso_counts = bcr_grp['BCR_CType'].value_counts()
        iso_total = iso_counts.sum()
        pct_igm = iso_counts.get('IGHM', 0) / iso_total * 100 if iso_total > 0 else np.nan
        pct_igg = iso_counts.get('IGHG', 0) / iso_total * 100 if iso_total > 0 else np.nan
        pct_iga = iso_counts.get('IGHA', 0) / iso_total * 100 if iso_total > 0 else np.nan
        
        results.append({
            'Stage': stage, 'donor': donor, 'tissue': tissue_val,
            'n_total': n_total, 'n_bcr': bcr_cells, 
            'pct_bcr': bcr_cells / n_total * 100,
            'n_unique_clones': n_unique, 'clonality': clonality,
            'pct_IgM': pct_igm, 'pct_IgG': pct_igg, 'pct_IgA': pct_iga,
            'pct_singleton': n_singleton / n_unique * 100 if n_unique > 0 else np.nan,
        })
    return pd.DataFrame(results)

bcr_liver = calc_donor_bcr_metrics(obs, 'Liver')
bcr_blood = calc_donor_bcr_metrics(obs, 'Blood')

for tissue_name, df in [('LIVER', bcr_liver), ('BLOOD', bcr_blood)]:
    print(f'\n{"─"*60}')
    print(f'{tissue_name} — Donor-level BCR metrics')
    print(f'{"─"*60}')
    summary = df.groupby('Stage', observed=True).agg(
        n_donors=('donor', 'nunique'),
        mean_bcr_cells=('n_bcr', 'mean'),
        mean_pct_bcr=('pct_bcr', 'mean'),
        mean_clonality=('clonality', 'mean'),
        mean_pct_singleton=('pct_singleton', 'mean'),
        mean_pct_IgM=('pct_IgM', 'mean'),
        mean_pct_IgG=('pct_IgG', 'mean'),
        mean_pct_IgA=('pct_IgA', 'mean'),
    ).round(2)
    # Reorder
    for stage_order in ['NL','IT','IA','AR','CR']:
        if stage_order in summary.index:
            row = summary.loc[stage_order]
            print(f'\n  {stage_order}: {int(row.n_donors)} donors, '
                  f'BCR cells={row.mean_bcr_cells:.0f}, '
                  f'clonality={row.mean_clonality:.4f}, '
                  f'singleton={row.mean_pct_singleton:.1f}%')
            print(f'    IgM={row.mean_pct_IgM:.1f}%, '
                  f'IgG={row.mean_pct_IgG:.1f}%, '
                  f'IgA={row.mean_pct_IgA:.1f}%')

In [ ]:
# Cell 5: Same for TCR
print(f'{"="*60}')
print('TCR COLUMNS DEEP ANALYSIS')
print(f'{"="*60}')

tcr_mask = obs['TCR_clone.id'].notna()
print(f'\nTCR+ cells: {tcr_mask.sum():,} / {len(obs):,} ({tcr_mask.mean()*100:.1f}%)')

# TCR by Stage × Tissue
print(f'\n--- TCR+ cells by Stage × Tissue ---')
tcr_xt = obs.groupby(['Stage', 'tissue'], observed=True).agg(
    total_cells=('TCR_clone.id', 'size'),
    tcr_cells=('TCR_clone.id', lambda x: x.notna().sum()),
).reset_index()
tcr_xt['pct'] = (tcr_xt['tcr_cells'] / tcr_xt['total_cells'] * 100).round(1)
print(tcr_xt.to_string(index=False))

# TCR V-gene usage top10 by Stage
print(f'\n--- Top 10 TCR V-genes by Stage ---')
for stage in ['NL','IT','IA','AR']:
    sub = obs[(obs['Stage']==stage) & obs['TCR_v_gene.x'].notna()]
    top = sub['TCR_v_gene.x'].value_counts().head(5)
    print(f'  {stage}: {dict(top)}')

In [ ]:
# Cell 6: Donor-level TCR metrics
print(f'{"="*60}')
print('DONOR-LEVEL TCR METRICS — IT-Oriented')
print(f'{"="*60}')

def calc_donor_tcr_metrics(obs_df, tissue_val):
    sub = obs_df[obs_df['tissue'] == tissue_val].copy()
    results = []
    for (stage, donor), grp in sub.groupby(['Stage', 'donor'], observed=True):
        n_total = len(grp)
        tcr_cells = grp['TCR_clone.id'].notna().sum()
        if tcr_cells == 0:
            results.append({
                'Stage': stage, 'donor': donor, 'tissue': tissue_val,
                'n_total': n_total, 'n_tcr': 0, 'pct_tcr': 0,
                'n_unique_clones': 0, 'clonality': 0,
                'pct_singleton': np.nan, 'max_clone_size': 0,
            })
            continue
        tcr_grp = grp[grp['TCR_clone.id'].notna()]
        clone_counts = tcr_grp['TCR_clone.id'].value_counts()
        n_unique = len(clone_counts)
        n_singleton = (clone_counts == 1).sum()
        max_clone = clone_counts.max()
        if n_unique > 1:
            freqs = clone_counts.values / clone_counts.sum()
            entropy = -np.sum(freqs * np.log2(freqs))
            max_entropy = np.log2(n_unique)
            clonality = 1 - (entropy / max_entropy) if max_entropy > 0 else 0
        else:
            clonality = 0
        results.append({
            'Stage': stage, 'donor': donor, 'tissue': tissue_val,
            'n_total': n_total, 'n_tcr': tcr_cells,
            'pct_tcr': tcr_cells / n_total * 100,
            'n_unique_clones': n_unique, 'clonality': clonality,
            'pct_singleton': n_singleton / n_unique * 100,
            'max_clone_size': max_clone,
        })
    return pd.DataFrame(results)

tcr_liver = calc_donor_tcr_metrics(obs, 'Liver')
tcr_blood = calc_donor_tcr_metrics(obs, 'Blood')

for tissue_name, df in [('LIVER', tcr_liver), ('BLOOD', tcr_blood)]:
    print(f'\n{"─"*60}')
    print(f'{tissue_name} — Donor-level TCR metrics')
    print(f'{"─"*60}')
    for stage in ['NL','IT','IA','AR','CR']:
        sub = df[df['Stage']==stage]
        if len(sub) == 0:
            print(f'  {stage}: no data')
            continue
        print(f'\n  {stage}: {len(sub)} donors')
        print(f'    TCR cells: {sub.n_tcr.mean():.0f} (mean), '
              f'clonality: {sub.clonality.mean():.4f}, '
              f'singleton: {sub.pct_singleton.mean():.1f}%, '
              f'max_clone: {sub.max_clone_size.mean():.0f}')

In [ ]:
# Cell 7: BCR + lineage cross-tab (which lineages have BCR data?)
print(f'{"="*60}')
print('BCR/TCR × LINEAGE × TISSUE × STAGE')
print(f'{"="*60}')

lineage_col = 'major_lineage'

# BCR by lineage
print('\n--- BCR+ cells by lineage ---')
bcr_by_lin = obs[obs['BCR_clone.id'].notna()].groupby(
    [lineage_col], observed=True
).size().sort_values(ascending=False)
print(bcr_by_lin.to_string())

# TCR by lineage
print('\n--- TCR+ cells by lineage ---')
tcr_by_lin = obs[obs['TCR_clone.id'].notna()].groupby(
    [lineage_col], observed=True
).size().sort_values(ascending=False)
print(tcr_by_lin.to_string())

# BCR by lineage × Stage × tissue
print('\n--- BCR+ cells by lineage × Stage × Tissue ---')
bcr_detail = obs[obs['BCR_clone.id'].notna()].groupby(
    ['Stage', 'tissue', lineage_col], observed=True
).size().reset_index(name='count')
bcr_pvt = bcr_detail.pivot_table(
    index=[lineage_col, 'tissue'], columns='Stage', values='count', fill_value=0
)
# Reorder columns
for col_order in [['NL','IT','IA','AR','CR']]:
    present = [c for c in col_order[0] if c in bcr_pvt.columns]
    bcr_pvt = bcr_pvt[present]
print(bcr_pvt.to_string())

In [ ]:
# Cell 8: BCR V-gene usage — IT vs NL comparison
print(f'{"="*60}')
print('BCR V-GENE USAGE: IT vs NL (IT-oriented)')
print(f'{"="*60}')

for tissue_val in ['Liver', 'Blood']:
    print(f'\n--- {tissue_val} ---')
    for stage in ['NL', 'IT', 'IA', 'AR']:
        sub = obs[(obs['Stage']==stage) & (obs['tissue']==tissue_val) & obs['BCR_v_gene'].notna()]
        if len(sub) == 0:
            print(f'  {stage}: no BCR data')
            continue
        vgene = sub['BCR_v_gene'].value_counts()
        total = vgene.sum()
        top5 = vgene.head(5)
        print(f'  {stage} (n={total}): ', end='')
        parts = [f'{g} {c} ({c/total*100:.1f}%)' for g, c in top5.items()]
        print(', '.join(parts))

# BCR Isotype by Stage × Tissue (% format)
print(f'\n{"="*60}')
print('BCR ISOTYPE DISTRIBUTION: Stage × Tissue')
print(f'{"="*60}')
for tissue_val in ['Liver', 'Blood']:
    print(f'\n--- {tissue_val} ---')
    for stage in ['NL', 'IT', 'IA', 'AR']:
        sub = obs[(obs['Stage']==stage) & (obs['tissue']==tissue_val) & obs['BCR_CType'].notna()]
        if len(sub) == 0:
            print(f'  {stage}: no data')
            continue
        iso = sub['BCR_CType'].value_counts()
        total = iso.sum()
        pcts = {k: f'{v/total*100:.1f}%' for k, v in iso.items()}
        print(f'  {stage} (n={total}): {pcts}')

In [ ]:
# Cell 9: Save all results to CSV for future reference
SAVE_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis-v2/BCR_TCR'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save donor-level BCR metrics
bcr_all = pd.concat([bcr_liver, bcr_blood], ignore_index=True)
bcr_all.to_csv(f'{SAVE_DIR}/donor_level_BCR_metrics.csv', index=False)
print(f'Saved: donor_level_BCR_metrics.csv ({len(bcr_all)} rows)')

# Save donor-level TCR metrics  
tcr_all = pd.concat([tcr_liver, tcr_blood], ignore_index=True)
tcr_all.to_csv(f'{SAVE_DIR}/donor_level_TCR_metrics.csv', index=False)
print(f'Saved: donor_level_TCR_metrics.csv ({len(tcr_all)} rows)')

# Save BCR isotype by stage × tissue
iso_pivot.to_csv(f'{SAVE_DIR}/BCR_isotype_by_stage_tissue.csv')
print(f'Saved: BCR_isotype_by_stage_tissue.csv')

# Save BCR × lineage detail
bcr_pvt.to_csv(f'{SAVE_DIR}/BCR_by_lineage_stage_tissue.csv')
print(f'Saved: BCR_by_lineage_stage_tissue.csv')

print(f'\n✅ All results saved to: {SAVE_DIR}')
print(f'\n{"="*60}')
print('SUMMARY: What we now know')
print(f'{"="*60}')
print(f'''
BCR data: {bcr_mask.sum():,} cells (5.3%) — 10 columns in h5ad
  Columns: clone.id, v_gene, j_gene, cdr3_nt, CType (isotype)
  Isotypes: IGHM, IGHG, IGHA, IGHD
  ⚠️ CR has ZERO BCR data (same as TCR)
  ⚠️ Heavy chain only (no light chain pairing)

TCR data: {tcr_mask.sum():,} cells (26.5%) — 5 columns in h5ad
  Columns: clone.id, v_gene.x, j_gene.x, cdr3_nt.x, CType
  CType: TRAC only (alpha chain)
  ⚠️ CR has ZERO TCR data

NEXT: Design IT-oriented donor-level BCR/TCR analysis
  1. Mann-Whitney NL→IT for clonality, isotype shift
  2. Tissue-separated (Liver vs Blood) patterns
  3. BCR V-gene usage IT-specific patterns
  4. Integration with B/PlasmaB subcluster annotation
''')